In [2]:
import pandas as pd 
import pandasql as ps                                                                                                                                                                                                         
                                                                                                                                                                                                                                
df = pd.read_excel('raw_data.xlsx')

df.head(100)

,Unique_ID,Amount,history_date,OpportunityProductGroup,Vertical,stage,close_date
0,OPP-46735909,13046,2024-06-25,Prod_001,Government,Discovery,2024-06-30
1,OPP-46735909,13046,2024-06-27,Prod_001,Government,Discovery,2024-09-25
2,OPP-46735909,13046,2024-09-09,Prod_001,Government,Discovery,2024-12-02
3,OPP-46735909,13046,2024-12-02,Prod_001,Government,Discovery,2025-07-01
4,OPP-46735909,13046,2024-12-10,Prod_001,Government,Closed Lost,2024-12-10
...,...,...,...,...,...,...,...
95,OPP-89488923,24030,2025-11-17,Prod_002,Business,Closed Lost,2025-11-17
96,OPP-11085735,63071,2025-04-08,Prod_001,Business,Meeting and Presentation,2025-06-30
97,OPP-11085735,63071,2025-04-15,Prod_001,Business,Meeting and Presentation,2025-05-14
98,OPP-11085735,63071,2025-04-16,Prod_001,Business,Proposal Delivery,2025-05-14


In [3]:
cleaned_df = ps.sqldf("""
         with cte as (
         select unique_id, amount, opportunityproductgroup, vertical, stage, history_date, close_date,
        rank() over(partition by unique_id, amount, opportunityproductgroup, vertical, stage order by history_date desc, close_date desc) as rank 
         from df
        )
         select distinct unique_id, amount, opportunityproductgroup, vertical, stage, history_date, close_date
         from cte
        where rank = 1
         """)   

cleaned_df.head()


,unique_id,amount,opportunityproductgroup,vertical,stage,history_date,close_date
0,OPP-10015184,44706,Prod_001,Business,Closed Lost,2026-01-06 00:00:00.000000,2026-01-06 00:00:00.000000
1,OPP-10015184,44706,Prod_001,Business,Discovery,2025-07-24 00:00:00.000000,2025-09-24 00:00:00.000000
2,OPP-10015184,44706,Prod_001,Business,Proposal Delivery,2025-12-15 00:00:00.000000,2026-02-25 00:00:00.000000
3,OPP-10043641,118822,Prod_002,Government,Closed Lost,2025-06-08 00:00:00.000000,2025-06-08 00:00:00.000000
4,OPP-10043641,118822,Prod_002,Government,Discovery,2025-06-05 00:00:00.000000,2025-06-08 00:00:00.000000


In [96]:
# total pipeline
ps.sqldf("""
         with cte as (
        select distinct unique_id, amount, history_date, stage, rank() over(partition by unique_id order by history_date) as rank
        from cleaned_df
         )

         select sum(amount), count(unique_id), count(distinct unique_id)
         from cte
         where rank = 1
         and stage != 'Closed Won'
         and stage != 'Closed Lost'
         """)   

,sum(amount),count(unique_id),count(distinct unique_id)
0,121834714,1760,1760


In [97]:
# total pipeline q1 2026
ps.sqldf("""
with cte as (                                                                                                                                      
      select distinct unique_id, amount, history_date, stage,                                                                                      
             rank() over(partition by unique_id order by history_date desc) as rank_desc,                                                            
             rank() over(partition by unique_id order by history_date) as rank_asc                                                                   
      from cleaned_df                                                                                                                                
      where history_date < '2026-01-01'                                                                                                              
  ),                                                        
  terminal as (                                                                                                                                      
      select unique_id                                      
      from cte                                                                                                                                       
      where rank_asc = 1
      and stage IN ('Closed Won','Closed Lost')                                                                                                      
  )                                                         
  select sum(amount), count(unique_id)
  from cte                                                                                                                                           
  where rank_desc = 1
  and stage NOT IN ('Closed Won','Closed Lost')                                                                                                      
  and unique_id not in (                                    
      select unique_id                                                                                                                               
      from terminal
      where unique_id is not null                                                                                                                    
  )           
         """)   

,sum(amount),count(unique_id)
0,18246359,203


In [98]:
# total pipeline q1 2026 business 
ps.sqldf("""
with cte as (                                                                                                                                      
      select distinct unique_id, amount, history_date, stage,                                                                                      
             rank() over(partition by unique_id order by history_date desc) as rank_desc,                                                            
             rank() over(partition by unique_id order by history_date) as rank_asc                                                                   
      from cleaned_df                                                                                                                                
      where history_date < '2026-01-01'   
      and vertical = 'Business'                                                                                                         
  ),                                                        
  terminal as (                                                                                                                                      
      select unique_id                                      
      from cte                                                                                                                                       
      where rank_asc = 1
      and stage IN ('Closed Won','Closed Lost')                                                                                                      
  )                                                         
  select sum(amount), count(unique_id)
  from cte                                                                                                                                           
  where rank_desc = 1
  and stage NOT IN ('Closed Won','Closed Lost')                                                                                                      
  and unique_id not in (                                    
      select unique_id                                                                                                                               
      from terminal
      where unique_id is not null                                                                                                                    
  )           
         """)   

,sum(amount),count(unique_id)
0,10660549,125


In [4]:
# total pipeline q1 2026 product 001
ps.sqldf("""
with cte as (                                                                                                                                      
      select distinct unique_id, amount, OpportunityProductGroup, history_date, stage,                                                                                      
             rank() over(partition by unique_id order by history_date desc) as rank_desc,                                                            
             rank() over(partition by unique_id order by history_date) as rank_asc                                                                   
      from cleaned_df                                                                                                                                
      where history_date < '2026-01-01'                                                                                                      
  ),                                                        
  terminal as (                                                                                                                                      
      select unique_id                                      
      from cte                                                                                                                                       
      where rank_asc = 1
      and stage IN ('Closed Won','Closed Lost')                                                                                                      
  )                                                         
  select sum(amount), count(unique_id)
  from cte                                                                                                                                           
  where rank_desc = 1
  and stage NOT IN ('Closed Won','Closed Lost')                                                                                                      
  and unique_id not in (                                    
      select unique_id                                                                                                                               
      from terminal
      where unique_id is not null                                                                                                                    
  )         
and OpportunityProductGroup = 'Prod_001'   
         """)   

,sum(amount),count(unique_id)
0,4147571,42


In [ ]:
# total closed
ps.sqldf("""
         with cte as (
        select distinct unique_id, amount, history_date, stage, rank() over(partition by unique_id order by history_date desc) as rank
        from cleaned_df
         )

         select sum(amount), count(unique_id), count(distinct unique_id)
         from cte
         where rank = 1
         and stage = 'Closed Won'
         """)   

,sum(amount),count(unique_id),count(distinct unique_id)
0,11046344,213,213


In [11]:
# total closed q1 2026
ps.sqldf("""
         with cte as (
        select distinct unique_id, amount, history_date, OpportunityProductGroup, vertical, stage, rank() over(partition by unique_id order by history_date desc) as rank
        from cleaned_df
        where history_date BETWEEN '2026-01-01' AND '2026-03-31'
        
        
         )

         select sum(amount), count(unique_id), count(distinct unique_id)
         from cte
         where rank = 1
         and stage = 'Closed Won'
         and history_date 
         and OpportunityProductGroup = 'Prod_001'  
         -- and vertical = 'Business'
         
         """)   

,sum(amount),count(unique_id),count(distinct unique_id)
0,251647,2,2


In [105]:
# typical deal worth
ps.sqldf("""
         with cte as (
        select distinct unique_id, amount, history_date, stage, rank() over(partition by unique_id order by history_date desc) as rank
        from cleaned_df
         )

         select avg(amount)
         from cte
         where rank = 1
         """)   

,avg(amount)
0,68437.763626


metric	amount	opportunities
total pipeline	£121,834,714	1760
total pipeline q1 2026	£18,246,359	203
total pipeline q1 2026 business 	£10,660,549	125
total pipeline q1 2026 product 001	£4,147,571	42
total closed	£11,046,344	213
total closed q1 2026	£919,065	8
total closed q1 2026 business	£833,473	6
total closed q1 2026 product 001	£251,647	2
avg deal worth 	£68,438	
